In [ ]:
import pandas as pd

# Load the cleaned election data
df = pd.read_csv("clean_2021.csv")

key_cols = [
    'state_ut_name', 'ac_no', 'ac_name', 'candidate_no', 'candidate_name',
    'party', 'category', 'sex', 'age', 'total', 'pct_votes_polled', 'total_electors'
]


df = df[key_cols]


df_sorted = df.sort_values(['state_ut_name', 'ac_no', 'total'], ascending=[True, True, False])


def get_winner_runner_up(group):
    winner = group.iloc[0]
    runner_up = group.iloc[1] if len(group) > 1 else None
    margin = winner['total'] - runner_up['total'] if runner_up is not None else winner['total']
    
    return pd.Series({
        'winner_name': winner['candidate_name'],
        'winner_party': winner['party'],
        'winner_votes': winner['total'],
        'runner_up_name': runner_up['candidate_name'] if runner_up is not None else None,
        'runner_up_party': runner_up['party'] if runner_up is not None else None,
        'runner_up_votes': runner_up['total'] if runner_up is not None else None,
        'vote_margin': margin,
        'total_votes_polled': group['total'].sum(),
        'num_candidates': len(group)
    })


constituency_results = df_sorted.groupby(
    ['state_ut_name', 'ac_no', 'ac_name'],
    group_keys=False
).apply(get_winner_runner_up).reset_index()

constituency_results['winner_vote_share_pct'] = constituency_results['winner_votes'] / constituency_results['total_votes_polled'] * 100
constituency_results['runner_up_vote_share_pct'] = constituency_results['runner_up_votes'] / constituency_results['total_votes_polled'] * 100

constituency_results['vote_margin_pct'] = constituency_results['vote_margin'] / constituency_results['total_votes_polled'] * 100


constituency_results = constituency_results.sort_values(['state_ut_name', 'ac_no']).reset_index(drop=True)

constituency_results.to_csv("constituency_results_2021.csv", index=False)


print(constituency_results.head())
print(constituency_results.columns.tolist())


  state_ut_name  ac_no             ac_name              winner_name  \
0   WEST BENGAL    1.0           MEKLIGANJ  ADHIKARY PARESH CHANDRA   
1   WEST BENGAL    2.0         MATHABHANGA            SUSHIL BARMAN   
2   WEST BENGAL    3.0    COOCHBEHAR UTTAR              SUKUMAR ROY   
3   WEST BENGAL    4.0  COOCHBEHAR DAKSHIN        NIKHIL RANJAN DEY   
4   WEST BENGAL    5.0          SITALKUCHI     BAREN CHANDRA BARMAN   

  winner_party  winner_votes        runner_up_name runner_up_party  \
0         AITC         99338          DADHIRAM RAY             BJP   
1          BJP        113249  GIRINDRA NATH BARMAN            AITC   
2          BJP        120483  BINAY KRISHNA BARMAN            AITC   
3          BJP         91560     AVIJIT DE BHOWMIK            AITC   
4          BJP        124955     PARTHA PRATIM RAY            AITC   

   runner_up_votes  vote_margin  total_votes_polled  num_candidates  \
0            84653        14685              198744              10   
1         

C:\Users\Admin\AppData\Local\Temp\ipykernel_24176\53124087.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(get_winner_runner_up).reset_index()
